In [1]:
pip install transformers torch

Note: you may need to restart the kernel to use updated packages.


DEPRECATION: vtk -PKG-VERSION has a non-standard version number. pip 24.0 will enforce this behaviour change. A possible replacement is to upgrade to a newer version of vtk or contact the author to suggest that they release a version with a conforming version number. Discussion can be found at https://github.com/pypa/pip/issues/12063


## Using DistillBert instead of BERT for speed and less compuatational load

In [7]:
import torch
from transformers import DistilBertTokenizer, DistilBertModel

In [3]:
import pandas as pd
import numpy as np
import tqdm

In [4]:
df = pd.read_csv("prepared_data.csv")

In [5]:
df

,movie,chapter,character,dialog,part,dialog_len
0,Harry Potter and the Philosopher's Stone,Doorstep Delivery,Albus Dumbledore,I should have known that you would be here...P...,1,10
1,Harry Potter and the Philosopher's Stone,Doorstep Delivery,Minerva McGonagall,"Good evening, Professor Dumbledore. Are the ru...",1,9
2,Harry Potter and the Philosopher's Stone,Doorstep Delivery,Albus Dumbledore,"I'm afraid so, Professor. The good, and the bad.",1,9
3,Harry Potter and the Philosopher's Stone,Doorstep Delivery,Minerva McGonagall,Do you think it wise to trust Hagrid with some...,1,14
4,Harry Potter and the Philosopher's Stone,Doorstep Delivery,Albus Dumbledore,"Ah, Professor, I would trust Hagrid with my life.",1,9
...,...,...,...,...,...,...
4417,Harry Potter and the Deathly Hallows Part 2,The Wizard's Choice,Harry Potter,"Well, it wasn't boring, was it?",8,6
4418,Harry Potter and the Deathly Hallows Part 2,Nineteen Years Later,Hermione Granger,Don't forget to give Professor Longbottom our ...,8,8
4419,Harry Potter and the Deathly Hallows Part 2,Nineteen Years Later,Harry Potter,There's nothing scary about thestrals. They're...,8,25
4420,Harry Potter and the Deathly Hallows Part 2,Nineteen Years Later,Harry Potter,Albus Severus Potter. You were named after two...,8,27


## Defining tokenizer and model instance

In [8]:
tokenizer = DistilBertTokenizer.from_pretrained('distilbert-base-uncased')
model = DistilBertModel.from_pretrained('distilbert-base-uncased')

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

In [12]:
def get_sentence_embedding(sentences,tokenizer=tokenizer,model=model):

    # the tokenizer object will convert all words/tokens from in each of the sentences from the input list  'sentences'  into tokenes.
    inputs = tokenizer(sentences, return_tensors='pt', padding=True, truncation=True)
    
    
    with torch.no_grad():
        outputs = model(**inputs)

    # I am accessing the embedding. the output from the model is an output object and has a lot of things/keys. The embedding is stored as the 'last_hidden_state'
    token_embeddings = outputs.last_hidden_state

    # becasue bert returns a vector of shape 768 for each token, each sentence embedding will  have shape (n,768) where n is the length of the sentence
    # which is why, I am taking the mean of each token's embedding to get the sentence embedding. The max pool will take the maximum value from each tokens embedding in the sentence returning
    # a single vector (768) for each sentence.

    sentence_embedding, _ = torch.mean(token_embeddings, dim=1)  

    return sentence_embedding

In [13]:
# creating a list of dialoges

dialoges = df['dialog'].to_list()

# now, i am applying the sentence embedding function to all the dialoges.

dialoges_embeddings =  get_sentence_embedding(dialoges)

KeyboardInterrupt: 